In [34]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder,StandardScaler,LabelEncoder,OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score,roc_auc_score,precision_score,recall_score
from collections import defaultdict
from sklearn.model_selection import GridSearchCV

In [35]:
df = pd.read_csv('data/WA_Fn-UseC_-Telco-Customer-Churn.xls')

In [36]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [37]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [38]:
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].mean())

In [39]:
# df['HasFamily'] = ((df['Partner'] == 'Yes') | (df['Dependents'] == 'Yes')).map({True: 'Yes', False: 'No'})

In [40]:
#df['Is_streaming'] = ((df['StreamingMovies'] == 'Yes') | (df['StreamingTV'] == 'Yes')).map({True:'Yes',False:'No'})

df['Is_streaming'] = np.where(
    (df['StreamingMovies'] == 'Nointernetservice') | (df['StreamingTV'] == 'Nointernetservice'), 'No',
        np.where(
            (df['StreamingMovies'] == 'Yes') | (df['StreamingTV'] == 'Yes'), 'Yes', 'No'))

In [41]:
# df['Monthly_charges_flag'] = np.where((df['MonthlyCharges']<=40),'LowMonthlyCharges',
#                                       np.where((df['MonthlyCharges']>40) & (df['MonthlyCharges'] <= 70),'MediumMonthlyCharges','HighMonthlyCharges'))

In [42]:
#df['Online_backup_security'] = ((df['OnlineBackup'] == 'Yes') | (df['OnlineSecurity'] == 'Yes')).map({True:"Yes",False:'No'})
# df['Online_backup_security'] = np.where(
#     (df['OnlineSecurity'] == 'Nointernetservice') | (df['OnlineBackup'] == 'Nointernetservice'), 'Nointernetservice',
#         np.where(
#             (df['OnlineSecurity'] == 'Yes') | (df['OnlineBackup'] == 'Yes'), 'Yes', 'No'))

In [43]:
# df['additional_services'] = np.where(((df['OnlineSecurity'] == 'Yes') | (df['OnlineBackup'] == 'Yes') | (df['DeviceProtection'] == 'Yes') | (df['TechSupport'] == 'Yes')),'Yes',
#                                      np.where(df['OnlineSecurity'] == 'Nointernetservice','Nointernetservice','No'))

In [44]:
drop_columns = ['customerID','StreamingMovies','StreamingTV','TotalCharges']
for col in drop_columns:
    if col in df.columns:
        df.drop(columns=[col],inplace=True)
#df.drop(columns=['customerID','Partner','Dependents','StreamingMovies','StreamingTV'],inplace=True)
df.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'Contract',
       'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'Churn',
       'Is_streaming'],
      dtype='str')

In [45]:
#df['PaymentMethod'] = (df['PaymentMethod'] == 'Electroniccheck').map({True:'Electroniccheck',False:'Other'})

In [46]:
df['MultipleLines'] = np.where((df['MultipleLines'] == 'No phone service') | (df['MultipleLines'] == 'No'),'No','Yes')
df['OnlineSecurity'] = np.where((df['OnlineSecurity'] == 'No phone service') | (df['OnlineSecurity'] == 'No'),'No','Yes')
df['OnlineBackup'] = np.where((df['OnlineBackup'] == 'No phone service') | (df['OnlineBackup'] == 'No'),'No','Yes')
df['DeviceProtection'] = np.where((df['DeviceProtection'] == 'No phone service') | (df['DeviceProtection'] == 'No'),'No','Yes')
df['TechSupport'] = np.where((df['TechSupport'] == 'No phone service') | (df['TechSupport'] == 'No'),'No','Yes')

In [47]:
# for col in categorical_feature:
#     df[col] =  df[col].str.replace(' ', '')
#     df[col] =  df[col].str.replace('-', '')

In [48]:
X = df.drop(columns=['Churn'])
y = df['Churn']
X

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,Is_streaming
0,Female,0,Yes,No,1,No,No,DSL,No,Yes,No,No,Month-to-month,Yes,Electronic check,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,One year,No,Mailed check,56.95,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,Month-to-month,Yes,Mailed check,53.85,No
3,Male,0,No,No,45,No,No,DSL,Yes,No,Yes,Yes,One year,No,Bank transfer (automatic),42.30,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,One year,Yes,Mailed check,84.80,Yes
7039,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,One year,Yes,Credit card (automatic),103.20,Yes
7040,Female,0,Yes,Yes,11,No,No,DSL,Yes,No,No,No,Month-to-month,Yes,Electronic check,29.60,No
7041,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,No


In [57]:
numerical_feature = X.select_dtypes(exclude="str").columns
categorical_feature = X.select_dtypes(include="str").columns
#ohe_categorical = ['']

oe_cat = ['InternetService','Contract','PaymentMethod']

categorical_feature = categorical_feature.drop(oe_cat, errors='ignore')

for col in categorical_feature:
    X[col] =  X[col].str.replace(' ', '')
    X[col] =  X[col].str.replace('-', '')
for col in oe_cat:
    X[col] =  X[col].str.replace(' ', '')
    X[col] =  X[col].str.replace('-', '')


In [58]:
print(oe_cat)
print(categorical_feature)

['InternetService', 'Contract', 'PaymentMethod']
Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'PaperlessBilling', 'Is_streaming'],
      dtype='str')


In [59]:
numerical_feature

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges'], dtype='str')

In [60]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)
print(X_train)
print(y_train)

      gender  SeniorCitizen  ... MonthlyCharges Is_streaming
1695    Male              0  ...          70.70          Yes
1095    Male              0  ...          80.55          Yes
3889    Male              0  ...          19.30           No
3667  Female              1  ...          96.55          Yes
2902  Female              1  ...          74.10           No
...      ...            ...  ...            ...          ...
3772    Male              0  ...          95.00          Yes
5191  Female              0  ...          91.10          Yes
5226    Male              0  ...          21.15           No
5390    Male              1  ...          99.45          Yes
860     Male              0  ...          19.80           No

[4930 rows x 17 columns]
1695     No
1095     No
3889     No
3667     No
2902     No
       ... 
3772    Yes
5191     No
5226     No
5390    Yes
860      No
Name: Churn, Length: 4930, dtype: str


In [61]:
X.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,Is_streaming
0,Female,0,Yes,No,1,No,No,DSL,No,Yes,No,No,Monthtomonth,Yes,Electroniccheck,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,Oneyear,No,Mailedcheck,56.95,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,Monthtomonth,Yes,Mailedcheck,53.85,No
3,Male,0,No,No,45,No,No,DSL,Yes,No,Yes,Yes,Oneyear,No,Banktransfer(automatic),42.30,No
4,Female,0,No,No,2,Yes,No,Fiberoptic,No,No,No,No,Monthtomonth,Yes,Electroniccheck,70.70,No


In [62]:
preprocessor = ColumnTransformer([
    ('OE',OrdinalEncoder(categories=[['No','DSL','Fiberoptic'],['Monthtomonth','Oneyear','Twoyear'],['Electroniccheck','Mailedcheck','Banktransfer(automatic)','Creditcard(automatic)']]),oe_cat),
    ('OHE',OneHotEncoder(drop = 'first'),categorical_feature),
    ('SC',StandardScaler(),numerical_feature)
])

In [63]:
le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)
print(y_train)

[0 0 0 ... 0 1 0]


In [64]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [65]:
X_train

array([[ 1.        ,  1.        ,  0.        , ..., -0.43683092,
         0.88107786,  0.19592677],
       [ 2.        ,  0.        ,  0.        , ..., -0.43683092,
        -1.28426262,  0.52275463],
       [ 0.        ,  1.        ,  3.        , ..., -0.43683092,
        -0.79399685, -1.50955058],
       ...,
       [ 0.        ,  0.        ,  0.        , ..., -0.43683092,
        -0.83485233, -1.44816666],
       [ 2.        ,  0.        ,  0.        , ...,  2.28921522,
        -0.83485233,  1.14986595],
       [ 0.        ,  1.        ,  3.        , ..., -0.43683092,
        -0.2628756 , -1.49296033]], shape=(4930, 17))

In [66]:
models = {
                "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
                "Random Forest": RandomForestClassifier(class_weight='balanced', n_estimators=200, random_state=42),
                "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42),
                "LightGBM": LGBMClassifier(class_weight='balanced', random_state=42),
                "CatBoost": CatBoostClassifier(verbose=0, random_state=42),
                "Gradient Boosting": GradientBoostingClassifier(random_state=42),
                "SVM": SVC(class_weight='balanced', probability=True, random_state=42),
                "KNN": KNeighborsClassifier(n_neighbors=5)
            }

In [67]:
if 'report' not in locals():
    report = defaultdict(list)


In [68]:
for model_name,model in models.items():

    model.fit(X_train,y_train)

    predict = model.predict(X_test)

    roc_auc = roc_auc_score(y_test,predict)*100
    f1__score = f1_score(y_test,predict)*100
    precision = precision_score(y_test,predict)*100
    recall = recall_score(y_test,predict)

    report[model_name].append((roc_auc, f1__score,recall,precision))

    #report[model_name] = roc_auc,f1__score

[LightGBM] [Info] Number of positive: 1295, number of negative: 3635
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000647 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 364
[LightGBM] [Info] Number of data points in the train set: 4930, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


c:\Users\Dhvanish\OneDrive\Desktop\ML projects\Customer churn prediction\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [69]:
report

defaultdict(list,
            {'Logistic Regression': [(77.50790707572905,
               64.14844267726971,
               0.8432055749128919,
               51.76470588235295)],
             'Random Forest': [(73.2438028223223,
               60.53268765133172,
               0.6533101045296167,
               56.390977443609025)],
             'XGBoost': [(71.0331044413201,
               58.157389635316704,
               0.5278745644599303,
               64.74358974358975)],
             'LightGBM': [(76.88400087843819,
               64.22413793103449,
               0.7787456445993032,
               54.64547677261614)],
             'CatBoost': [(70.96388215344142,
               58.160237388724035,
               0.5121951219512195,
               67.27688787185355)],
             'Gradient Boosting': [(71.88958167777166,
               59.628543499511245,
               0.5313588850174216,
               67.92873051224944)],
             'SVM': [(77.911411319627,
           

In [70]:

# for model, scores in sorted(report.items(), key=lambda item: item[1], reverse=True):
#     print(f"{model} -> roc_auc_score: {scores[0]:.4f}")
#     print(f"{model} -> f1_score: {scores[1]}")
#     print("\n")

table_data = []
for model_name, scores in report.items():
    # Fetch old and new scores safely
    old_roc, old_f1,old_recall,old_precision = scores[0] if len(scores) >= 4 else (None, None,None,None)
    new_roc, new_f1,new_recall,new_precision = scores[-1] if scores else (None, None,None,None)
    
    # table_data.append({
    #     'Model Name': model_name,
    #     'Old ROC-AUC': old_roc,
    #     'New ROC-AUC': new_roc,
    #     'Old F1': old_f1,
    #     'New F1': new_f1
    # })
    table_data.append({
        'Model Name': model_name,
        # 'Old ROC-AUC': old_roc,
        'ROC-AUC': new_roc,
        # 'Old F1': old_f1,
        'F1': new_f1,
        'Recall': new_recall,
        'Precision': new_precision
    })

# Convert to DataFrame and display
df_results = pd.DataFrame(table_data)
print(df_results)  # Or just type 'df_results' if in a Jupyter notebook cell


            Model Name    ROC-AUC         F1    Recall  Precision
0  Logistic Regression  77.507907  64.148443  0.843206  51.764706
1        Random Forest  73.243803  60.532688  0.653310  56.390977
2              XGBoost  71.033104  58.157390  0.527875  64.743590
3             LightGBM  76.884001  64.224138  0.778746  54.645477
4             CatBoost  70.963882  58.160237  0.512195  67.276888
5    Gradient Boosting  71.889582  59.628543  0.531359  67.928731
6                  SVM  77.911411  65.132497  0.813589  54.302326
7                  KNN  69.464707  55.565611  0.534843  57.815443


In [116]:

# 1. Format the scores as strings "ROC / F1" for clean rows
formatted_report = {}
for model_name, scores in report.items():
    formatted_report[model_name] = [f"{roc:.4f} / {f1:.4f}" for roc, f1 in scores]

# 2. Create the DataFrame (this automatically sets model names as column headers)
# We use pd.DataFrame.from_dict with orient='columns' (default)
df_transposed = pd.DataFrame.from_dict(formatted_report, orient='columns')

# 3. Label the row index to represent the experiment runs
df_transposed.index = [f"Run {i+1}" for i in range(len(df_transposed))]

# Display the DataFrame
df_transposed


,Logistic Regression,Random Forest,XGBoost,LightGBM,CatBoost,Gradient Boosting,SVM,KNN
Run 1,78.1873 / 65.2115,73.0253 / 60.2762,71.0788 / 58.1681,75.7191 / 62.8986,71.6711 / 59.3103,71.8232 / 59.5661,77.8803 / 65.0069,70.1173 / 56.5492
Run 2,78.1548 / 65.1670,73.0460 / 60.3434,69.2820 / 55.3520,76.2210 / 63.4146,71.2370 / 58.5700,71.6268 / 59.2666,78.1727 / 65.4141,70.0848 / 56.4982
Run 3,78.1327 / 65.1639,73.2735 / 60.6908,69.3041 / 55.3846,76.5888 / 63.9594,71.1174 / 58.3741,71.7140 / 59.4059,77.9882 / 65.1842,70.0627 / 56.4706
Run 4,77.5079 / 64.1484,73.2438 / 60.5327,71.0331 / 58.1574,76.8840 / 64.2241,70.9639 / 58.1602,71.8896 / 59.6285,77.9114 / 65.1325,69.4647 / 55.5656


In [85]:
param_grid = {'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l1','l2'], 'solver': ['liblinear']}
grid = GridSearchCV(LogisticRegression(class_weight='balanced'),scoring='f1',cv=10,param_grid=param_grid)

grid.fit(X_train,y_train)

grid_predict = grid.predict(X_test)

grid_f1 = f1_score(y_test,grid_predict)

print(grid.best_score_)
print(grid.best_params_)

c:\Users\Dhvanish\OneDrive\Desktop\ML projects\Customer churn prediction\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Dhvanish\OneDrive\Desktop\ML projects\Customer churn prediction\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\Dhvanish\OneDrive\Desktop\ML projects\Customer churn prediction\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed 

0.6154495769233799
{'C': 100, 'penalty': 'l1', 'solver': 'liblinear'}
